# Sequence Context Analysis

**Does the local nucleotide sequence surrounding an edited adenosine differ from the background?**

This notebook addresses the following questions:

- Which nucleotides are enriched or depleted around edited adenosines?
- Are specific 3-mer, 5-mer, 7-mer, or longer sequence motifs preferred?
- Does the sequence context of edited adenosines differ from comparable unedited adenosines?
- Does sequence context differ between highly and weakly edited sites?

# Imports & Define Project Paths

In [3]:
import numpy as np 
import pandas as pd 
from pathlib import Path
import matplotlib.pyplot as plt 

# Define the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Define important project folders
DATA_DIR = PROJECT_ROOT / "data" 
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

# Create output directories if they don't already exist
    # parents=True: auto create missing directories
    # exist_ok=True: no error is raised if they already exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Print path to verify the project structure
print("Root:", PROJECT_ROOT)
print("Data:", DATA_DIR)
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)

Root: /home/mescalin/bioinf/rna_editing_project
Data: /home/mescalin/bioinf/rna_editing_project/data
Results: /home/mescalin/bioinf/rna_editing_project/results
Figures: /home/mescalin/bioinf/rna_editing_project/figures


# Load REDIportal dataset

In [4]:
rediportal_file = DATA_DIR / "TABLE1_hg38_v3.txt"

# Read the tab-separated file into a pandas DataFrame
editing_df = pd.read_csv(rediportal_file, sep="\t")

# Display the dimension of the dataset
n_rows, n_cols = editing_df.shape

print("Number of rows aka. editing sites: ", n_rows)
print("Number of columns aka. features: ", n_cols)

# Display the first 5 rows
editing_df.head()

Number of rows aka. editing sites:  749926
Number of columns aka. features:  43


,Accession,Region,Position,Ref,Ed,Strand,db,type,dbsnp,repeat,...,rnacentral,nTCGASamples,nTCGAPrj,sTCGAprj,sTCGAdiseasetype,nTCGAdiseasetype,PlotsTCGA,PRIDE,REDInet_BS,REDInet_mean
0,EDHSAAAA0000,chr1,87158,T,C,-,A,ALU,-,SINE/AluJo,...,-,0.0,0.0,0,0,0.0,"0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,...",-,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-...",-1.0
1,EDHSAAAA0001,chr1,87168,T,C,-,A,ALU,-,SINE/AluJo,...,-,0.0,0.0,0,0,0.0,"0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,...",-,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-...",-1.0
2,EDHSAAAA0002,chr1,87171,T,C,-,A,ALU,-,SINE/AluJo,...,-,2.0,2.0,101000000000000000000,100000000001000000000000,2.0,"0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,...",-,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-...",-1.0
3,EDHSAAAA0003,chr1,87189,T,C,-,A,ALU,-,SINE/AluJo,...,-,0.0,0.0,0,0,0.0,"0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,...",-,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-...",-1.0
4,EDHSAAAA0004,chr1,87218,T,C,-,A,ALU,-,SINE/AluJo,...,-,1.0,1.0,10000000000,100000000000000000000000,1.0,"0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,...",-,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-...",-1.0


# Explore the dataset

- Accession: A unique Accession Number
- Region: Chromosome Coordinate
- Position: Chromosome
- Ref: Reference Nucleotide
- Ed: Edited Nucleodite
- Strand: Strand (+ or -)
- db: indicating whether the site was already known in previous editing databases (RADAR and DARNED)
- type: Editing location: ALU, REP, NONREP
- dbsnp: reference ID for dbSNP
- repeat: Class and family of repeat including the RNA editing position.
- Func: Function
- Gene: Gene name
- ExonicFunc: Exonic function according to gencode v45
- AAChange: RNA editing sites residing in protein coding regions and affecting codon integrity
- phastConsElements100way: ?? PhastCons conservation scores calculated for multiple alignments of 45 vertebrate genomes to the human genome; first number = Score (ranges from 0 (no conservation) to 1000 (max conservation)), second number = ?? 
- nSamples: number of GTEx samples 
- nTissues: number of GTEx tissues 
- sTissue: encoding of GTEx tissues 
- sBody: encoding of GTEx Body sites
- nBodySites: number of GTEx body sites
- PlotsBody: values (min, Q1, median, Q3, max) to plot the boxplot for each bodysite (GTEx)
- geneID: gene ID (ensembl)
- geneType: type of gene (ensembl)
- geneTag: Annotation Attributes (ensembl - https://www.gencodegenes.org/pages/tags.html)
- uniprotID: ID number from UniProt (database for protein sequence and functional information)
- rnacentral: ID for RNA central (but no result in RNA central)
- nTCGASamples: number of TCGA Samples
- nTCGAPrj: number of TCGA Studies 
- sTCGAprj: encoding of TCGA Studies 
- sTCGAdiseasetype: encoding of TCGA Diseases 
- nTCGAdiseasetype: number of TCGA Diseases
- PlotsTCGA: values (min, Q1, median, Q3, max) to plot the boxplot for each bodysite (TCGA)
- PRIDE: PRoteomics IDEntifications database entry
- REDInet_BS: REDInet Pvalue per Body Site
- REDInet_mean: mean of REDInet Pvalue over all Body Sites (-1 means no sample)


ANNOVAR for Gene, Func, ExonicFunc, AAChange,
- wgEncodeGencodeBasicV45: Gencode v45 Annotation linked to GeneCards (GENCODE)
- refGene: RefSeq Annotation (RefSeq database)
- knownGene: UCSC Annotation (UCSC Known Gene database)


Details <br>
AA change:
  - Gene Symbol
  - Ensembl Transcript ID
  - specific Exon where the edit occurs.
  - The Coding DNA change. c.A100G: unedited nt, position, edited nt
  - The Protein change. p.V861A: unedited AA, position, edited AA
  - example: PITRM1:ENST00000451104.6:exon22:c.T2582C:p.V861A,PITRM1:ENST00000678436.1:exon23:c.T2678C:p.V893A


not sure how nSamples and nTCGASamples are

In [5]:
# Display column names
print("Column names: \n", editing_df.columns.tolist(), "\n")

# Show data types and missing values
print("Data types and missing values: \n")
editing_df.info()

Column names: 
 ['Accession', 'Region', 'Position', 'Ref', 'Ed', 'Strand', 'db', 'type', 'dbsnp', 'repeat', 'Func.wgEncodeGencodeBasicV45', 'Gene.wgEncodeGencodeBasicV45', 'ExonicFunc.wgEncodeGencodeBasicV45', 'AAChange.wgEncodeGencodeBasicV45', 'Func.refGene', 'Gene.refGene', 'ExonicFunc.refGene', 'AAChange.refGene', 'Func.knownGene', 'Gene.knownGene', 'ExonicFunc.knownGene', 'AAChange.knownGene', 'phastConsElements100way', 'nSamples', 'nTissues', 'sTissue', 'sBody', 'nBodySites', 'PlotsBody', 'geneID', 'geneType', 'geneTag', 'uniprotID', 'rnacentral', 'nTCGASamples', 'nTCGAPrj', 'sTCGAprj', 'sTCGAdiseasetype', 'nTCGAdiseasetype', 'PlotsTCGA', 'PRIDE', 'REDInet_BS', 'REDInet_mean'] 

Data types and missing values: 

<class 'pandas.DataFrame'>
RangeIndex: 749926 entries, 0 to 749925
Data columns (total 43 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Accession                           

# Filter high-confidence A-to-I sites

A-to-I editing normally appears as :
- A to G on the (+) strand
- T to C on the (-) strand

The (-) strand sites must later be reverse-complemented => extrancted window is an RNA sequence with A at its center.

In [13]:
# Filter canonical A-to-I RNA editing sites

# Work on a copy
sequence_df = editing_df.copy() 

# Display the observed reference-to-edited nucleotide combinations.
# This helps verify how REDIportal stores editing events on each strands
editing_changes = (
    sequence_df
    .groupby(["Strand", "Region", "Ref", "Ed"], dropna=False) # group identical rows by strand, ref, ed. rows w missing values not discarded
    .size() # count how many sites are in each group (n_sites)
    .reset_index(name="n_sites") # convert the result back into a normal DataFrame, names "n_sites"
    .sort_values("n_sites", ascending=False) # sort the combinations in  descending order
)

editing_changes


,Strand,Region,Ref,Ed,n_sites
0,+,chr1,A,G,393982
1,-,chr1,T,C,355944


### Observations

The dataset shows 2 types of edit sites:
1. A-to-G on the (+) strand (direct edit sites)
2. T-to-C on the (-) strand (complementary edit sites)

The A-to-G sites actually represent A-to-I edited sites performed by ADAR in eukaryote cells. Inosines are interpreted as guanosine during sequencing, hence the change.
A-to-I edits occur approximately in the same amount on both positive and negative strands.

In [ ]:
# Filter the canonical events

# Define canonical A-to-I events in genomic orientation


# Obtain genomic sequences

# Extract k-mer windows

# Nucleotide frequency analysis

# Sequence logos

# 3-mer enrichment

# 5-mer enrichment

# 7-mer enrichment

# 15-mer enrichment

# Save processed dataset